In [1]:
import torch
import vitlab
from vitlab.activations import ActivationReader
from vitlab.datasets import get_splits
from vitlab.sae import load_layer_sae

DEVICE = "cuda"
BACKBONE_CKPT = "../runs/dinov2-base_full_epochs30/best"
SAE_CKPT = "../saes/dinov2_fitz_l9_topk16_d3072"
DATASET = "fitzpatrick17k"

model  = vitlab.load_model(BACKBONE_CKPT, device=DEVICE)
reader = ActivationReader(model.backbone)
sae = load_layer_sae(SAE_CKPT, device=DEVICE)

train, _, _ = get_splits(DATASET, model_key=model.spec.key)
px = train[0]["pixel_values"].unsqueeze(0).to(DEVICE)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/7976 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1140 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2281 [00:00<?, ?it/s]

In [11]:
with torch.no_grad():
    acts    = reader.read(px, sae.spec.site)
    print(f"Activations Shape (Batch, Sequence, Dimension): {acts.shape}")
    patches = acts[:, model.spec.n_prefix_tokens:, :]
    print(f"Patches Shape (Batch, Patches, Dimension): {patches.shape}")
    codes   = sae.encode(patches.reshape(-1, patches.shape[-1]))
    print(f"Codes Shape (Batch x Sequence, SAE Dimension): {codes.shape}")

active = (codes > 0).sum(-1).float().mean()
print(f"{codes.shape[1]} concepts, {active:.1f} active per patch (should be ~top_k)")

'blocks.9.resid_post'

In [2]:
sae.spec

SAESpec(site='blocks.9.resid_post', model_key='dinov2-base', sae_class='TopKSAE', nb_concepts=3072, d_model=768, top_k=16, normalizer='zscore', dataset='', token_select='patches', num_train_tokens=2041600, final_r2=0.7823613559554717, final_dead_frac=0.1471354216337204, epochs=30, extra={'sae_type': 'TopKSAE', 'normalize': 'zscore'})